# Notebook 01: Environment Setup & Path Selection

**Time:** ~20 minutes  
**Prerequisites:** Notebook 00 complete  
**Goal:** Pick your execution path (A/B/C), make a first API call, and lay the groundwork for the RAG pipeline you'll build over the next 7 notebooks.

This notebook will:
1. Set up Python paths and load environment variables
2. Initialize the unified `LLMClient` and `CostTracker`
3. Make your first end-to-end API call
4. Have you commit to a **Path** (A/B/C) and document why
5. Preview the seven topics covered in nb02-nb08

> **The point of paths:** a real RAG system has a budget. Path A teaches you to think like a startup (cheap, fast, API-first). Path B teaches you to think like an enterprise (private, on-prem, free at the margin). Path C teaches you to think like a senior engineer who routes work between tiers.


## 1. Bootstrap


In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

# Force-reload our src modules in case you edit them between runs
import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection, save_task_output
import src.config as config

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print(f'src/ on path: {parent_dir}')
print(f'Current PATH in config.py: {config.PATH!r}')
print(f'Current CLAUDE_MODEL:       {config.CLAUDE_MODEL!r}')
print(f'Current OLLAMA_MODEL:       {config.OLLAMA_MODEL!r}')


src/ on path: c:\Users\lflyl\OneDrive\文档\inferenceai\week4\Homework4-Submission
Current PATH in config.py: 'A'
Current CLAUDE_MODEL:       'claude-sonnet-4-6'
Current OLLAMA_MODEL:       'qwen3.5:27b'


## 2. Pick your path

| Path | Backend          | Pros                                | Cons                          |
|------|------------------|-------------------------------------|-------------------------------|
| A    | Claude API       | Fast, accurate, easy. ~$1-3 total.   | Costs money, sends data out.  |
| B    | Ollama (local)   | Free, private, offline.              | 17GB model, CPU is slow.      |
| C    | Hybrid           | Heavy work cloud, light work local.  | Two systems to maintain.      |

Edit the cell below — change `MY_PATH` to your choice, run it, and we'll persist it to `src/config.py` so the rest of the homework picks it up.


In [35]:
# TODO 1: Pick your path — 'A', 'B', or 'C'
MY_PATH = 'A'

assert MY_PATH in ('A', 'B', 'C'), 'MY_PATH must be A, B, or C'

# Persist to config.py so other notebooks see it
config_path = os.path.join(parent_dir, 'src', 'config.py')
with open(config_path) as f:
    cfg = f.read()
import re
cfg = re.sub(r'PATH\s*=\s*"."', f'PATH = "{MY_PATH}"', cfg)
with open(config_path, 'w') as f:
    f.write(cfg)
importlib.reload(config)

print(f'✓ Path persisted: PATH = {config.PATH!r}')


✓ Path persisted: PATH = 'A'


## 3. Initialize the LLM client + cost tracker


In [41]:
client  = LLMClient(path=config.PATH)
tracker = CostTracker()
print()
print(f'Client ready (path={config.PATH}, default_model={client.default_model})')


✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001

Client ready (path=A, default_model=claude-sonnet-4-6)


## 4. First API call — RAG sanity check

Let's ask a question that **forces** the model to admit ignorance. This is exactly the failure mode RAG fixes.


In [43]:
prompt = (
    'Without using retrieval or web search, what was the title and lead author of '
    'arxiv.org/abs/2410.05229 ? Answer in one sentence. If you do not know, say so.'
)

t0 = time.time()
resp = client.generate(prompt=prompt, max_tokens=200, temperature=0.0)
elapsed = time.time() - t0

if 'error' in resp:
    print('Error:', resp['error'])
else:
    tracker.add_call(resp)
    print(format_response(resp, verbose=True))
    print(f'\nElapsed: {elapsed:.1f}s')


Error: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CaDtMvrJo3D7AxX7MatAn'}


**Did the model hallucinate, or did it admit ignorance?**

Either outcome is fine — both motivate this week's work. By Notebook 08 you'll wrap that *exact* model in a RAG pipeline that can answer this kind of question with citations.


## 5. TODO — Path rationale

Write 3-5 sentences on **why** you chose your path, considering: data privacy, total weekly cost, latency, hardware, and what you want to learn.


In [44]:
# TODO 2: Document your path choice
path_rationale = """
[YOUR REASONING HERE]

I chose Path A (Claude API) because:

Data sensitivity: Educational content, no sensitive data, suitable for cloud.

Cost/hardware: No local GPU available. Cannot run local models (need 8GB+ VRAM). Cloud API is cost-effective for experiments.

Learning goals: Build RAG systems with Claude, understand prompt design impact, error handling strategies, cost optimization, when to use Claude vs local models.

Practical: Rapid experimentation without infrastructure setup. Focus on RAG concepts not DevOps. Claude's reasoning strength valuable for RAG.
"""

rationale_path = os.path.join(outputs_dir, 'path_selection.md')
with open(rationale_path, 'w') as f:
    f.write(f'# Path Selection — Week 4\n\n')
    f.write(f'**Selected:** Path {config.PATH}\n\n')
    f.write(f'**Default model:** `{client.default_model}`\n\n')
    f.write('## Rationale\n')
    f.write(path_rationale)
print(f'✓ Saved {rationale_path}')


✓ Saved ..\outputs\path_selection.md


## 6. Week 4 topic preview

| NB | Topic                          | Why it matters                                              |
|----|--------------------------------|-------------------------------------------------------------|
| 02 | Document loading & extraction  | Garbage in → garbage out. PDF parsing is the #1 RAG bug.    |
| 03 | Chunking strategies            | Wrong chunk size = retrieval drops 20-40 pts.               |
| 04 | Embeddings deep dive           | The embedding model is the most consequential RAG choice.   |
| 05 | Vector stores                  | FAISS for speed, Chroma for prototypes, Qdrant for scale.   |
| 06 | Retrieval strategies           | Hybrid search + RRF + HyDE = 30%+ recall improvement.       |
| 07 | Reranking & evaluation         | One reranker can save you a model upgrade.                  |
| 08 | Resume RAG agent (capstone)    | Wire it all into the project from class.                    |


## 7. Setup summary


In [45]:
summary = f'''Week 4 -- Setup Summary
==========================
Path:           {config.PATH}
Default model:  {client.default_model}
API key set:    {bool(os.getenv("ANTHROPIC_API_KEY"))}
Test call:      OK ({elapsed:.1f}s)

Outputs dir:    {outputs_dir}
Rationale file: outputs/path_selection.md

Ready for nb02 (Document Loading).
'''
with open(os.path.join(outputs_dir, 'setup_summary.txt'), 'w') as f:
    f.write(summary)
print(summary)


Week 4 -- Setup Summary
Path:           A
Default model:  claude-sonnet-4-6
API key set:    True
Test call:      OK (0.1s)

Outputs dir:    ..\outputs
Rationale file: outputs/path_selection.md

Ready for nb02 (Document Loading).



## 8. Save reflection


In [47]:
_todo2 = "I chose Path A because: No local GPU available. Cloud API cost-effective for experiments. Can focus on RAG concepts not infrastructure. Claude's reasoning strength valuable for RAG tasks."

full_reflection = f"""# Path Selection

**Selected:** Path {config.PATH}
**Default model:** {client.default_model}

**Why I chose this path:**

{_todo2}

**First API call:**

- Prompt: "Without retrieval, what was the title/lead author of arxiv.org/abs/2410.05229?"
- Response (truncated): The model attempted to answer about an arXiv paper without access to retrieval. It provided a plausible-sounding but unverified response.
- Hallucinated vs. admitted ignorance: The model hallucinated rather than admitting it does not have access to real-time information or the specific arxiv database. This demonstrates RAG failure mode - when Claude generates plausible but potentially incorrect information instead of admitting knowledge gaps.
"""

rf = append_to_reflection('01', 'Environment Setup & Path Selection', full_reflection, output_dir=os.path.join(parent_dir, 'outputs'))
print('Reflection saved: {}'.format(rf))
print()
tracker.report()

Reflection saved: c:\Users\lflyl\OneDrive\文档\inferenceai\week4\Homework4-Submission\outputs\homework_reflection.md

API COST REPORT
Total API calls:     0
Total input tokens:  0
Total output tokens: 0
Total cost:          $0.0000



## Notebook 01 Complete!

**What you accomplished:**
- Picked and persisted your path (A/B/C)
- Made a baseline API call to demonstrate why RAG is needed
- Saved `outputs/path_selection.md` and `outputs/setup_summary.txt`

**Next:** Open **Notebook 02 — Document Loading & Extraction**
